## Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

## Load Preprocessed Dataset

In [2]:
processed_path = "../dataset/processed"
df = pd.read_parquet(processed_path)

## Map target to binary

In [3]:
df['default_flag'] = df['loan_status'].apply(lambda x: 1 if x == 'Charged Off' else 0)

## Define features

In [4]:
numeric_features = ["revol_util", "dti", "total_bal_ex_mort", "num_accts_ever_120_pd", "delinq_2yrs", "annual_inc"]
categorical_features = ["home_ownership", "purpose", "grade", "sub_grade", "term", "verification_status"]

df[numeric_features] = df[numeric_features].fillna(0.0)

## Preprocessing

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

X = preprocessor.fit_transform(df[numeric_features + categorical_features])
y = df['default_flag'].values

## Handle Class Imbalance (Undersample Majority)

In [6]:
df_full = pd.DataFrame(X)
df_full['target'] = y

# Separate majority and minority
df_majority = df_full[df_full.target == 0]
df_minority = df_full[df_full.target == 1]

# Undersample majority
df_majority_downsampled = resample(df_majority,
                                   replace=False,
                                   n_samples=len(df_minority)*100,  # keep 100x minority
                                   random_state=42)

df_balanced = pd.concat([df_majority_downsampled, df_minority])

X_balanced = df_balanced.drop('target', axis=1).values
y_balanced = df_balanced['target'].values

## Train/Test Split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced, y_balanced, test_size=0.2, random_state=42, stratify=y_balanced
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train target distribution: {np.bincount(y_train)}, Test target distribution: {np.bincount(y_test)}")

Train shape: (1535, 69), Test shape: (384, 69)
Train target distribution: [1520   15], Test target distribution: [380   4]


## Build Neural Network

In [8]:
input_dim = X_train.shape[1]

model = Sequential([
    Dense(128, input_dim=input_dim, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])

early_stop = EarlyStopping(monitor='val_auc', patience=5, mode='max', restore_best_weights=True)

/Users/parikshitnarang/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:86: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


## Train the model

In [9]:
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    class_weight={0: 1, 1: 100},  # emphasize minority class
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - AUC: 0.3743 - loss: 1.7303 - val_AUC: 0.4656 - val_loss: 0.2855
Epoch 2/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.5223 - loss: 1.4658 - val_AUC: 0.7270 - val_loss: 0.4049
Epoch 3/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 980us/step - AUC: 0.7420 - loss: 1.4607 - val_AUC: 0.8156 - val_loss: 0.4228
Epoch 4/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 976us/step - AUC: 0.8105 - loss: 0.9289 - val_AUC: 0.8328 - val_loss: 0.3836
Epoch 5/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 933us/step - AUC: 0.7646 - loss: 0.6952 - val_AUC: 0.8615 - val_loss: 0.3601
Epoch 6/50


/Users/parikshitnarang/Library/Python/3.9/lib/python/site-packages/keras/src/callbacks/early_stopping.py:156: UserWarning: Early stopping conditioned on metric `val_auc` which is not available. Available metrics are: AUC,loss,val_AUC,val_loss
  current = self.get_monitor_value(logs)


39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 989us/step - AUC: 0.8476 - loss: 0.7748 - val_AUC: 0.8820 - val_loss: 0.2953
Epoch 7/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 930us/step - AUC: 0.9264 - loss: 0.5688 - val_AUC: 0.8934 - val_loss: 0.2418
Epoch 8/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - AUC: 0.9037 - loss: 0.8182 - val_AUC: 0.8902 - val_loss: 0.2304
Epoch 9/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 949us/step - AUC: 0.9811 - loss: 0.4583 - val_AUC: 0.8893 - val_loss: 0.2397
Epoch 10/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 938us/step - AUC: 0.9772 - loss: 0.5790 - val_AUC: 0.8934 - val_loss: 0.2098
Epoch 11/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 952us/step - AUC: 0.9636 - loss: 0.3879 - val_AUC: 0.8852 - val_loss: 0.1859
Epoch 12/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 920us/step - AUC: 0.9846 - loss: 0.3915 - val_AUC: 0.8902 - val_loss: 0.1661
Epoch 13/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 934us/step - AUC: 0.8961 - loss: 0.2457 - val_AUC: 0.8959 - val_loss: 0.1510
Epoch 14/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 957us/step - AUC: 0.965

## Evaluate the model

In [10]:
results = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {results[0]:.4f}, Test AUC: {results[1]:.4f}")

Test Loss: 0.2318, Test AUC: 0.4474


## Predict probabilities

In [11]:
y_pred_prob = model.predict(X_test)
y_pred_class = (y_pred_prob >= 0.5).astype(int)

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 439us/step


## Save the trained model (Keras 3 compatible)

In [12]:
# Option 1: Keras native format (.keras) — recommended
model.save("lending_club_nn_model.keras")
print("Model saved as 'lending_club_nn_model.keras'.")

# Option 2: HDF5 format (.h5)
model.save("lending_club_nn_model.h5")
print("Model saved as 'lending_club_nn_model.h5'.")


Model saved as 'lending_club_nn_model.keras'.
Model saved as 'lending_club_nn_model.h5'.
